# Step 7: compare tuned and untuned rankers

Use the same complete training data, fixed trip folds, feature list and metric rules for both configurations. The untuned model uses XGBoost's installed defaults, with only objective, metric, histogram algorithm, seed and CPU count specified. It is not the previously tuned Step 6 model.

The tuned model uses the configuration frozen by the 50-trial search. Its full-row CV score is still subject to hyperparameter-selection bias. Student-test results are computed only after that configuration is frozen, and are not used to change it.

This notebook regenerates tuned OOF probabilities and fits final models. Stacking and HMM results require their separate model artifacts; missing artifacts are reported explicitly.

In [1]:
%run 06_model_helpers.ipynb
import xgboost as xgb
from time import perf_counter
from datetime import datetime
from zoneinfo import ZoneInfo
import matplotlib.pyplot as plt

search_output = Path("outputs/step7_xgb_sample10000")
output_root = Path("results/step7_xgb")
output_root.mkdir(parents=True, exist_ok=True)
n_jobs = 8
data, points = load_model_data()
metric_names = ["top1", "macro_f1", "precision", "recall", "log_loss", "candidate_coverage"]

In [2]:
def fit_ranker(frame, params):
    sizes = frame.groupby("group_id")["label"].agg(["size", "sum"])
    eligible = sizes.index[(sizes["size"] >= 2) & (sizes["sum"] == 1)]
    training = frame[frame["group_id"].isin(eligible)]
    assert training["group_id"].is_monotonic_increasing
    groups = training.groupby("group_id", sort=False).size().to_numpy()
    model = xgb.XGBRanker(objective="rank:ndcg", eval_metric="ndcg@1", tree_method="hist",
                          random_state=seed, n_jobs=n_jobs, **params)
    model.fit(training[feature_columns], training["label"], group=groups, verbose=False)
    return model


## Full-row CV, OOF and final refits

Train on all eligible groups in the other four folds. Predict all valid held-out candidate rows, including singleton and no-positive groups. Save OOF scores as float64 so pooling does not introduce extra probability-rounding ties.

Each completed variant saves its CV metrics, OOF table and final model. Reuse a finished variant when rerunning the comparison; unfinished variants are rerun. The saved params, row count, predictor list and fold scheme must match.

In [3]:
def run_variant(name, params):
    folder = output_root / name
    folder.mkdir(parents=True, exist_ok=True)
    signature = {"params": params, "seed": seed, "rows": len(data), "features": feature_columns,
                 "fold_scheme": contract["fold_scheme"], "candidate_settings": contract["candidate_settings"]}
    signature_file = folder / "run_config.json"
    if signature_file.exists():
        assert json.loads(signature_file.read_text()) == signature
    signature_file.write_text(json.dumps(signature, indent=2) + "\n")
    required = ["cv_metrics.csv", "pooled_metrics.json", "timings.json", "model.json", f"oof_{name}.csv"]
    if all((folder / path).exists() for path in required):
        print("Reusing completed variant:", name, flush=True)
        return folder

    probability_oof = np.full(len(data), np.nan, dtype=np.float64)
    raw_oof = np.full(len(data), np.nan, dtype=np.float32)
    records = []
    for fold in range(5):
        training = data[data["fold"] != fold]
        validation = data[data["fold"] == fold]
        valid_points = points[points["fold"] == fold]
        assert set(training["trip_id"]).isdisjoint(validation["trip_id"])
        start = perf_counter()
        model = fit_ranker(training, params)
        fit_s = perf_counter() - start
        start = perf_counter()
        raw = model.predict(validation[feature_columns])
        probability = point_softmax(raw, validation["group_id"])
        predict_s = perf_counter() - start
        probability_oof[validation.index] = probability
        raw_oof[validation.index] = raw
        records.append({"fold": fold, **score_points(validation, probability, valid_points),
                        "fit_s": fit_s, "predict_s": predict_s})
        pd.DataFrame(records).to_csv(folder / "progress.csv", index=False)
        print(name, "fold", fold, "Top-1", records[-1]["top1"], flush=True)

    assert np.isfinite(probability_oof).all() and np.isfinite(raw_oof).all()
    oof = data[["group_id", "segment_id", "fold"]].copy()
    oof["score"], oof["raw_score"] = probability_oof, raw_oof
    oof.to_csv(folder / f"oof_{name}.csv", index=False)
    cv = pd.DataFrame(records)
    cv.to_csv(folder / "cv_metrics.csv", index=False)
    pooled = score_points(data, probability_oof, points)
    assert np.array_equal(np.sum([r["confusion_matrix"] for r in records], axis=0), pooled["confusion_matrix"])
    (folder / "pooled_metrics.json").write_text(json.dumps(pooled, indent=2) + "\n")
    start = perf_counter()
    model = fit_ranker(data, params)
    final_fit_s = perf_counter() - start
    model.save_model(folder / "model.json")
    (folder / "booster_config.json").write_text(model.get_booster().save_config())
    timings = {"final_fit_s": final_fit_s, "fit_s_mean": cv["fit_s"].mean(),
               "predict_s_mean": cv["predict_s"].mean(), "xgboost_version": xgb.__version__}
    (folder / "timings.json").write_text(json.dumps(timings, indent=2) + "\n")
    return folder

## Freeze selection before viewing student-test scores

The two final models are fit only on training data. Prediction uses the test predictor table's feature columns while labels are still blank. Reference labels are attached afterwards for scoring. This prevents those labels from entering candidate generation, features or prediction.

In [4]:
frozen = json.loads((search_output / "best_params.json").read_text())
assert frozen["completed_trials"] >= 50
variants = {"untuned": {}, "tuned_" + frozen["search_mode"]: frozen["params"]}
folders = {name: run_variant(name, params) for name, params in variants.items()}

test = pd.read_parquet("outputs/student_test/features.parquet")
assert test["label"].isna().all()
nearest = test["distance"] - test["distance_above_nearest_m"]
test["distance_ratio"] = ((test["distance"] + 1) / (nearest + 1)).astype("float32")
test_points = pd.read_csv("outputs/student_test/points.csv", dtype={"trip_id": str})
assert set(points["trip_id"]).isdisjoint(test_points["trip_id"])
valid_points = test_points[~test_points["bad_gps"]].copy()
valid_rows = test["group_id"].isin(valid_points["group_id"])
comparisons = []
for name, folder in folders.items():
    model = xgb.XGBRanker(n_jobs=n_jobs)
    model.load_model(folder / "model.json")
    start = perf_counter()
    raw = model.predict(test[feature_columns])
    probability = point_softmax(raw, test["group_id"])
    predict_s = perf_counter() - start
    predictions = test[["group_id", "segment_id"]].copy()
    predictions["score"], predictions["raw_score"] = probability, raw
    predictions.to_csv(folder / "student_test_predictions.csv", index=False)

    scoring = test.loc[valid_rows, ["group_id", "segment_id"]].copy()
    true_ids = scoring["group_id"].map(valid_points.set_index("group_id")["true_arc_id"])
    scoring["label"] = scoring["segment_id"].eq(true_ids).astype("int8")
    metrics = score_points(scoring, probability[valid_rows.to_numpy()], valid_points)
    (folder / "student_test_metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
    cv = pd.read_csv(folder / "cv_metrics.csv")
    pooled = json.loads((folder / "pooled_metrics.json").read_text())
    timings = json.loads((folder / "timings.json").read_text())
    row = {"variant": name, "params": json.dumps(variants[name]), "seed": seed,
           "radius_m": contract["candidate_settings"]["radius_m"], "K": contract["candidate_settings"]["K"],
           "fold_scheme": contract["fold_scheme"], "fit_s_mean": timings["fit_s_mean"],
           "predict_s_mean": timings["predict_s_mean"], "final_fit_s": timings["final_fit_s"],
           "final_predict_s": predict_s, "output_dir": str(folder), "test_status": "scored_once_after_selection"}
    for metric in metric_names:
        row[f"cv_{metric}_mean"] = cv[metric].mean()
        row[f"cv_{metric}_std"] = cv[metric].std()
        row[f"pooled_{metric}"] = pooled[metric]
        row[f"test_{metric}"] = metrics[metric]
    row["pooled_confusion_matrix"] = json.dumps(pooled["confusion_matrix"])
    row["test_confusion_matrix"] = json.dumps(metrics["confusion_matrix"])
    comparisons.append(row)

comparison = pd.DataFrame(comparisons)
comparison.to_csv(output_root / "comparison.csv", index=False)
display(comparison[["variant", "cv_top1_mean", "cv_top1_std", "pooled_top1", "test_top1", "test_log_loss"]])

Reusing completed variant: untuned


tuned_sample10000 fold 0 Top-1 0.5122744309578773


tuned_sample10000 fold 1 Top-1 0.8567627813331415


tuned_sample10000 fold 2 Top-1 0.921101181206124


tuned_sample10000 fold 3 Top-1 0.8304110515384244


tuned_sample10000 fold 4 Top-1 0.8961171414001603


,variant,cv_top1_mean,cv_top1_std,pooled_top1,test_top1,test_log_loss
0,untuned,0.803060,0.166174,0.804805,0.888413,0.302737
1,tuned_sample10000,0.803333,0.166419,0.805081,0.891493,0.310644


## Experiment ledger and downstream handoff

Append one row per variant to the shared experiment ledger. Trial history and the frozen selection remain in `outputs/step7_xgb_*`.

The tuned OOF file uses the same keys and folds as the other base models, so it can replace the old ranker input once LR, KNN and tree OOF files are available. It must not be mixed with a combiner trained on the old ranker scores.

A stacker and HMM cannot be evaluated here without their implementations and model inputs. Record those missing dependencies without inventing scores. Do not fit a temperature from student-test labels.

In [5]:
timestamp = datetime.now(ZoneInfo("Asia/Kolkata"))
ledger = comparison.copy()
ledger["run_id"] = ["step7_" + name + "_" + timestamp.strftime("%Y%m%d_%H%M%S") for name in ledger["variant"]]
ledger["timestamp"] = timestamp.isoformat()
ledger["model"] = "xgb_ranker"
ledger["search_output"] = str(search_output)
ledger["search_groups"] = frozen["search_groups"]
ledger["search_cv_top1"] = frozen["search_cv_top1"]
ledger["selection_bias_note"] = "Tuned CV follows selection on these folds; independent student-test metrics are the holdout check."
experiments = Path("results/experiments.csv")
existing = pd.read_csv(experiments) if experiments.exists() else pd.DataFrame()
pd.concat([existing, ledger], ignore_index=True).to_csv(experiments, index=False)
status = {"ranker_comparison": "complete", "stacking": "pending LR/KNN/tree OOF files and final models",
          "old_stacker": "not available", "HMM": "See results/hmm/ranker and results/final_comparison.csv for the latest decoder results",
          "temperature": "1.0; no temperature fitted on student-test labels",
          "tuned_oof": str(folders["tuned_" + frozen["search_mode"]] / ("oof_tuned_" + frozen["search_mode"] + ".csv"))}
(output_root / "downstream_status.json").write_text(json.dumps(status, indent=2) + "\n")
print("Student-test Top-1 change (percentage points):", 100 * (comparison.iloc[1]["test_top1"] - comparison.iloc[0]["test_top1"]))
print("Saved comparison and experiment rows. No further test-driven tuning is performed.")

Student-test Top-1 change (percentage points): 0.30793319415448606
Saved comparison and experiment rows. No further test-driven tuning is performed.
